# Disneyland RAG System Demo

This notebook demonstrates the RAG system for answering questions about Disneyland visitor reviews.

## Setup

Load environment, instantiate embeddings and LLM.

In [1]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from rag.config import DATA_PATH, EMBEDDING_MODEL_NAME, LLM_MODEL_NAME
from rag.embeddings import SentenceTransformerEmbeddings
from rag.ingest import load_reviews
from rag.vectorstore import get_or_build_collection
from rag.chain import ask
from langchain_litellm import ChatLiteLLM

print(f"Data path: {DATA_PATH}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"LLM model: {LLM_MODEL_NAME}")

/home/syaramionak/Projects/rag-system-poc/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
12:43:13 - LiteLLM:WARNING: common_utils.py:979 - litellm: could not pre-load bedrock-runtime response stream shape — Bedrock event-stream decoding will be unavailable. Error: No module named 'botocore'
12:43:13 - LiteLLM:WARNING: common_utils.py:24 - litellm: could not pre-load sagemaker-runtime response stream shape — SageMaker event-stream decoding will be unavailable. Error: No module named 'botocore'


Data path: /home/syaramionak/Projects/rag-system-poc/data/DisneylandReviews.csv
Embedding model: all-MiniLM-L6-v2
LLM model: litellm_proxy/openrouter/openai/gpt-4.1-mini


## Load and Embed Data

This cell loads reviews from CSV and embeds them into ChromaDB.
On first run, this takes 3-5 minutes. Subsequent runs load from disk instantly.

In [2]:
# Load reviews
print("Loading reviews from CSV...")
documents = load_reviews(DATA_PATH)
print(f"Loaded {len(documents)} reviews")

# Initialize embeddings
print(f"\nInitializing embeddings with {EMBEDDING_MODEL_NAME}...")
embeddings = SentenceTransformerEmbeddings()

# Build or load ChromaDB collection
print("Building/loading ChromaDB collection...")
collection = get_or_build_collection(documents, embeddings)
print(f"Collection size: {collection.count()} documents")

Loading reviews from CSV...
Loaded 42656 reviews

Initializing embeddings with all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12119.21it/s]


Building/loading ChromaDB collection...
Collection size: 6656 documents


## Initialize LLM

Create a ChatLiteLLM instance pointing to the LiteLLM proxy.

In [3]:
import os

proxy_url = os.getenv("LITELLM_PROXY_URL", "https://litellm.gke-prod.linnovate.net")
api_key = os.getenv("LITELLM_MASTER_KEY")

print(f"Proxy URL: {proxy_url}")
print(f"API Key: {'***' if api_key else 'NOT SET'}")

llm = ChatLiteLLM(
    model=LLM_MODEL_NAME,
    api_base=proxy_url,
    api_key=api_key,
    timeout=120,
)
print("\nLLM initialized successfully")

Proxy URL: https://litellm.gke-prod.linnovate.net
API Key: ***

LLM initialized successfully


## Example 1: Australia visitors at HongKong

Question: "What do visitors from Australia say about Disneyland in HongKong?"

In [10]:
question_1 = "What do visitors from Australia say about Disneyland in HongKong?"
print(f"Question: {question_1}")
print("\n" + "="*60)

answer_1 = ask(question_1, collection, embeddings, llm, auto_extract_filters=True, n_results=3)
print("Answer:")
print(answer_1)

Question: What do visitors from Australia say about Disneyland in HongKong?

Answer:
The visitor from Australia highly recommends visiting Hong Kong Disneyland, stating that it brings joy and happiness around every corner. They also mentioned that the lines to rides flowed well and the employees were super friendly.


## Example 2: Spring season visits

Question: "Is spring a good time to visit Disneyland?"

In [11]:
question_2 = "Is spring a good time to visit Disneyland?"
print(f"Question: {question_2}")
print("\n" + "="*60)

answer_2 = ask(question_2, collection, embeddings, llm, auto_extract_filters=True, n_results=3)
print("Answer:")
print(answer_2)

Question: Is spring a good time to visit Disneyland?

Answer:
Based on the reviews provided, there is no specific information about visiting Disneyland in the spring (March to May) except for one mention of early May. A visitor from Malaysia noted that early May was slightly crowded because many people take a long leave due to Labour Day, and they recommended going early in the morning to avoid the heat and crowds. Therefore, spring might be more crowded compared to other times like January or November, and it could be advisable to visit early in the day.


## Example 3: California in June

Question: "Is Disneyland California usually crowded in June?"

In [12]:
question_3 = "Is Disneyland California usually crowded in June?"
print(f"Question: {question_3}")
print("\n" + "="*60)

answer_3 = ask(question_3, collection, embeddings, llm, auto_extract_filters=True, n_results=3)
print("Answer:")
print(answer_3)

Question: Is Disneyland California usually crowded in June?

Answer:
The reviews do not provide information about how crowded Disneyland California is in June. They only mention that Disneyland Hong Kong was not crowded on a Tuesday in May compared to Disneyland California standards.


## Example 4: Staff friendliness in Paris

Question: "Is the staff in Paris friendly?"

In [13]:
question_4 = "Is the staff in Paris friendly?"
print(f"Question: {question_4}")
print("\n" + "="*60)

answer_4 = ask(question_4, collection, embeddings, llm, auto_extract_filters=True, n_results=3)
print("Answer:")
print(answer_4)

Question: Is the staff in Paris friendly?

Answer:
The reviews provided do not contain any information about the friendliness of the staff in Disneyland Paris. Therefore, I cannot answer this question based on the given reviews.


## Debug: Inspect Retrieved Documents

For a given question, see what documents are retrieved before they go to the LLM.

In [16]:
from rag.filter_parser import extract_filters
from rag.retriever import retrieve

debug_question = "Is the staff in Paris friendly?"
print(f"Debug question: {debug_question}")

# Extract filters
filters = extract_filters(debug_question, llm)
print(f"\nExtracted filters: {filters}")

# Retrieve documents
retrieved_docs = retrieve(
    debug_question,
    collection,
    embeddings,
    n_results=15,
    branch=filters.get("branch"),
    reviewer_location=filters.get("reviewer_location"),
    season=filters.get("season"),
)
print(f"\nRetrieved {len(retrieved_docs)} documents:")
print("\n" + "-"*60 + "\n")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"Document {i}:")
    print(f"  Metadata: {doc.metadata}")
    print(f"  Text: {doc.page_content[:200]}...")
    print()

Debug question: Is the staff in Paris friendly?

Extracted filters: {'branch': None, 'reviewer_location': None, 'season': None}

Retrieved 15 documents:

------------------------------------------------------------

Document 1:
  Metadata: {'reviewer_location': 'Malaysia', 'year_month': '2016-3', 'rating': 5, 'season': 'spring', 'review_id': '365852926', 'branch': 'Disneyland_HongKong'}
  Text: This is a place that everyone must go in their lifetime! The staffs are friendly and you get to forget your troubles....

Document 2:
  Metadata: {'season': 'autumn', 'rating': 5, 'review_id': '638701393', 'branch': 'Disneyland_HongKong', 'reviewer_location': 'Hong Kong', 'year_month': '2018-10'}
  Text: God I love this place.  Not as big as the Paris one but just as much fun.  The entertainment is spot on and very well organised.  Buy a fast pass though....

Document 3:
  Metadata: {'reviewer_location': 'United Kingdom', 'branch': 'Disneyland_HongKong', 'season': 'autumn', 'rating': 5, 'review_